# Web Scraping data
v4 - 
10/06/2026


In [1]:
import asyncio
import json
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse
 
import httpx
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
# ─── Configuration ────────────────────────────────────────────────────────────
 
# ⚠️ Update this path to where your Excel file lives
EXCEL_FILE = Path("/Users/anamoura/Desktop/SMAnalictics/DataExtraction/Bieberchella_urls2.xlsx")

OUTPUT_DIR = Path("output3")
OUTPUT_DIR.mkdir(exist_ok=True)
 
# One output set per sheet — named after the sheet
def sheet_paths(sheet_name: str) -> dict:
    safe = sheet_name.replace(" ", "_")
    return {
        "jsonl":      OUTPUT_DIR / f"{safe}_raw.jsonl",
        "parquet":    OUTPUT_DIR / f"{safe}_articles.parquet",
        "failed":     OUTPUT_DIR / f"{safe}_failed.txt",
        "checkpoint": OUTPUT_DIR / f"{safe}_checkpoint.json",
    }

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}
 
CONCURRENCY        = 2   # parallel async requests
DELAY_BETWEEN      = 10  # seconds between requests
PLAYWRIGHT_TIMEOUT = 25  # seconds
HTTP_TIMEOUT       = 20  # seconds

print('✅ Configuration ready')
print(f'   Excel file : {EXCEL_FILE}')
print(f'   Output dir : {OUTPUT_DIR.resolve()}')

✅ Configuration ready
   Excel file : /Users/anamoura/Desktop/SMAnalictics/DataExtraction/Bieberchella_urls2.xlsx
   Output dir : /Users/anamoura/Desktop/SMAnalictics/DataExtraction/output3


In [3]:
# ─── Load all sheets from Excel ───────────────────────────────────────────────

def load_sheets_from_excel(excel_path: Path) -> dict[str, list[str]]:
    """Returns {sheet_name: [url, ...]} for every sheet in the workbook."""
    if not excel_path.exists():
        raise FileNotFoundError(f"File not found: {excel_path}")

    xl = pd.ExcelFile(excel_path)
    sheets = {}

    for sheet in xl.sheet_names:
        df = xl.parse(sheet)

        # Auto-detect URL column
        url_col = None
        for col in df.columns:
            if str(col).strip().lower() in ("url", "link", "links", "urls", "href"):
                url_col = col
                break

        if url_col is None:
            print(f"  ⚠️  Sheet '{sheet}': no URL column found (columns: {list(df.columns)}) — skipping")
            continue

        urls = (
            df[url_col]
            .dropna()
            .astype(str)
            .str.strip()
            .pipe(lambda s: s[s.str.startswith("http")])
            .unique()
            .tolist()
        )
        sheets[sheet] = urls
        print(f"  📄 Sheet '{sheet}': {len(urls)} URLs")

    return sheets


SHEETS = load_sheets_from_excel(EXCEL_FILE)
print(f"\n✅ Loaded {sum(len(v) for v in SHEETS.values())} URLs across {len(SHEETS)} sheets")

  📄 Sheet '28Mar_10April': 64 URLs
  📄 Sheet '11_April_19April': 67 URLs
  📄 Sheet '20April_03May': 69 URLs

✅ Loaded 200 URLs across 3 sheets


In [7]:
# Load URLs
# Preview loaded sheets
for sheet_name, urls in SHEETS.items():
    print(f"\n📅 {sheet_name}: {len(urls)} URLs")
    for u in urls[:3]:
        print(f"  {u}")


📅 28Mar_10April: 64 URLs
  https://www.yahoo.com/entertainment/music/article/justin-bieber-is-headlining-coachella-this-month-how-the-pop-star-is-forging-a-new-path-163753630.html
  https://www.theguardian.com/music/2026/apr/10/coachella-2026-justin-bieber-sabrina-carpenter-karol-g
  https://www.rollingstone.com/music/music-news/justin-bieber-coachella-sneak-preview-1235540010/

📅 11_April_19April: 67 URLs
  https://www.newyorker.com/culture/the-lede/justin-bieber-pop-musics-fallen-angel-rises-again-at-coachella
  https://theconversation.com/justin-biebers-coachella-performance-wasnt-lazy-and-actually-references-50-years-of-music-history-280463
  https://www.theguardian.com/music/2026/apr/12/justin-bieber-coachella-review

📅 20April_03May: 69 URLs
  https://www.refinery29.com/en-us/justin-bieber-coachella-weekend-2026
  https://www.theatlantic.com/culture/2026/04/justin-bieber-coachella-performances-youtube/686871/
  https://www.forbes.com/sites/hughmcintyre/2026/04/24/justin-biebers-

In [8]:
# ─── Checkpoint helpers (per sheet) ───────────────────────────────────────────

def load_checkpoint(paths: dict) -> set:
    if paths["checkpoint"].exists():
        data = json.loads(paths["checkpoint"].read_text())
        return set(data.get("done", []))
    return set()

def save_checkpoint(done: set, paths: dict):
    paths["checkpoint"].write_text(json.dumps({"done": list(done)}))

def append_jsonl(record: dict, paths: dict):
    with paths["jsonl"].open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print('✅ Checkpoint helpers ready')

✅ Checkpoint helpers ready


In [9]:
# ─── Helpers ──────────────────────────────────────────────────────────────────
 
def is_allowed_by_robots(url: str) -> bool:
    try:
        parsed = urlparse(url)
        robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
        rp = RobotFileParser()
        rp.set_url(robots_url)
        rp.read()
        return rp.can_fetch("*", url)
    except Exception:
        return True
 
# ── Consent wall detection ────────────────────────────────────────────────────
BLOCK_PAGE_TITLES = {
    "your privacy choices", "access to this page has been denied",
    "just a moment", "attention required", "pardon our interruption",
    "please verify you are a human", "before you continue", "consent",
    "this website is unavailable in your location",
}

def is_block_page(record: dict) -> bool:
    title = (record.get("title") or "").strip().lower()
    return any(b in title for b in BLOCK_PAGE_TITLES)

# ── Date recovery ─────────────────────────────────────────────────────────────
def extract_date_from_url(url: str) -> str | None:
    patterns = [
        r"/(20\d{2})/(0[1-9]|1[0-2])/(0[1-9]|[12]\d|3[01])/",
        r"/(20\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])-",
        r"-(20\d{2})-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])[/-]",
    ]
    for pat in patterns:
        m = re.search(pat, url)
        if m:
            try:
                dt = datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)), tzinfo=timezone.utc)
                return dt.isoformat()
            except Exception:
                continue
    return None

def extract_date_from_text(text: str) -> str | None:
    if not text:
        return None
    month_map = {
        "january":"01","february":"02","march":"03","april":"04",
        "may":"05","june":"06","july":"07","august":"08",
        "september":"09","october":"10","november":"11","december":"12",
        "jan":"01","feb":"02","mar":"03","apr":"04",
        "jun":"06","jul":"07","aug":"08","sep":"09",
        "oct":"10","nov":"11","dec":"12",
    }
    patterns = [
        (r"\b(January|February|March|April|May|June|July|August|September|October|November|December|"
         r"Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\.?\s+(\d{1,2}),?\s+(20\d{2})\b", "named"),
        (r"\b(20\d{2})-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])\b", "iso"),
    ]
    for pat, fmt in patterns:
        m = re.search(pat, text[:3000], re.IGNORECASE)
        if m:
            try:
                g = m.groups()
                if fmt == "iso":
                    dt = datetime(int(g[0]), int(g[1]), int(g[2]), tzinfo=timezone.utc)
                else:
                    month = month_map.get(g[0].lower().rstrip("."))
                    if not month: continue
                    dt = datetime(int(g[2]), int(month), int(g[1]), tzinfo=timezone.utc)
                return dt.isoformat()
            except Exception:
                continue
    return None

def recover_date(record: dict) -> dict:
    if record.get("published_date"):
        record["date_source"] = "meta"
    else:
        date = extract_date_from_url(record["url"])
        if date:
            record["published_date"] = date
            record["date_source"] = "recovered_url"
        else:
            date = extract_date_from_text(record.get("text", ""))
            if date:
                record["published_date"] = date
                record["date_source"] = "recovered_text"
            else:
                record["date_source"] = "missing"
    return record

# ── HTML parser ───────────────────────────────────────────────────────────────
def parse_html(html: str, url: str, time_period: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
        tag.decompose()
    title = soup.title.string.strip() if soup.title else None
    article = (
        soup.find("article")
        or soup.find(attrs={"class": re.compile(r"article|content|body|story", re.I)})
        or soup.find("main")
        or soup.body
    )
    text = article.get_text(separator=" ", strip=True) if article else ""
    text = re.sub(r"\s+", " ", text)[:10000]
    def meta(name):
        tag = soup.find("meta", attrs={"name": name}) or soup.find("meta", attrs={"property": name})
        return tag.get("content", "").strip() if tag else None
    record = {
        "url": url,
        "domain": urlparse(url).netloc,
        "time_period": time_period,
        "title": title,
        "description": meta("description") or meta("og:description"),
        "author": meta("author") or meta("article:author"),
        "published_date": meta("article:published_time") or meta("og:article:published_time"),
        "keywords": meta("keywords"),
        "text": text,
        "text_length": len(text),
        "scraped_at": datetime.now(timezone.utc).isoformat(),
        "method": None,
        "status": None,
        "error": None,
    }
    return recover_date(record)

print('✅ All helpers ready (consent wall + date recovery + time_period tag)')

✅ All helpers ready (consent wall + date recovery + time_period tag)


In [10]:
# ─── Phase 1: Classification ───────────────────────────────────────────────────
 
async def classify_urls(urls: list[str], time_period: str) -> dict:
    results = {"ok": [], "blocked": [], "robots_blocked": [], "error": []}
    print(f"\n📋 [{time_period}] Phase 1: Classifying {len(urls)} URLs...")
 
    async with httpx.AsyncClient(headers=HEADERS, follow_redirects=True, timeout=10) as client:
        for url in urls:
            if not is_allowed_by_robots(url):
                results["robots_blocked"].append(url)
                print(f"  🤖 robots.txt blocked: {url}")
                continue
            try:
                r = await client.head(url)
                if r.status_code in (200, 301, 302):
                    results["ok"].append(url)
                    print(f"  ✅ {r.status_code} {url}")
                elif r.status_code in (403, 401, 429):
                    results["blocked"].append(url)
                    print(f"  🚫 {r.status_code} {url}")
                else:
                    results["error"].append(url)
                    print(f"  ⚠️  {r.status_code} {url}")
            except Exception as e:
                results["error"].append(url)
                print(f"  ❌ error: {url} — {e}")
            await asyncio.sleep(0.2)
 
    print(f"  Summary → OK: {len(results['ok'])} | Blocked: {len(results['blocked'])} | "
          f"robots.txt: {len(results['robots_blocked'])} | Error: {len(results['error'])}")
    return results
 

In [11]:
# ─── Phase 2: Async HTTP Scraping ─────────────────────────────────────────────
 
async def scrape_http(url: str, client: httpx.AsyncClient, semaphore: asyncio.Semaphore, time_period: str) -> dict:
    async with semaphore:
        try:
            r = await client.get(url, timeout=HTTP_TIMEOUT)
            record = parse_html(r.text, url, time_period)
            record["method"] = "httpx"
            record["status"] = r.status_code
            return record
        except Exception as e:
            return {
                "url": url, "domain": urlparse(url).netloc, "time_period": time_period,
                "title": None, "description": None, "author": None,
                "published_date": None, "keywords": None, "text": None,
                "text_length": 0, "scraped_at": datetime.now(timezone.utc).isoformat(),
                "method": "httpx", "status": None, "error": str(e), "date_source": "missing",
            }
 
async def run_http_phase(urls, done, paths, time_period):
    pending = [u for u in urls if u not in done]
    if not pending:
        print(f"  ⏭️  [{time_period}] All HTTP URLs already done.")
        return [], []
    print(f"\n🌐 [{time_period}] Phase 2: HTTP scraping {len(pending)} URLs...")
    semaphore = asyncio.Semaphore(CONCURRENCY)
    succeeded, failed = [], []
    async with httpx.AsyncClient(headers=HEADERS, follow_redirects=True) as client:
        tasks = [scrape_http(u, client, semaphore, time_period) for u in pending]
        for i, coro in enumerate(asyncio.as_completed(tasks)):
            record = await coro
            await asyncio.sleep(DELAY_BETWEEN)
            if record.get("error") or not record.get("text") or is_block_page(record):
                failed.append(record["url"])
                reason = "CONSENT WALL" if is_block_page(record) else "FAILED"
                print(f"  ❌ [{i+1}/{len(pending)}] {reason} — {record['url']}")
            else:
                succeeded.append(record)
                done.add(record["url"])
                append_jsonl(record, paths)
                save_checkpoint(done, paths)
                print(f"  ✅ [{i+1}/{len(pending)}] {record['title'] or record['url'][:60]}")
    print(f"  Done — {len(succeeded)} succeeded, {len(failed)} failed")
    return succeeded, failed

In [12]:
# ─── Phase 3: Playwright for JS-heavy / failed sites ─────────────────────────
 
async def scrape_playwright(url: str, page, time_period: str) -> dict:
    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=PLAYWRIGHT_TIMEOUT * 1000)
        await asyncio.sleep(2)
        html = await page.content()
        record = parse_html(html, url, time_period)
        record["method"] = "playwright"
        record["status"] = 200
        return record
    except Exception as e:
        return {
            "url": url, "domain": urlparse(url).netloc, "time_period": time_period,
            "title": None, "description": None, "author": None,
            "published_date": None, "keywords": None, "text": None,
            "text_length": 0, "scraped_at": datetime.now(timezone.utc).isoformat(),
            "method": "playwright", "status": None, "error": str(e), "date_source": "missing",
        }
 
async def run_playwright_phase(urls, done, paths, time_period):
    pending = [u for u in urls if u not in done]
    if not pending:
        print(f"  ⏭️  [{time_period}] No Playwright URLs to process.")
        return [], []
    try:
        from playwright.async_api import async_playwright
    except ImportError:
        print("  ⚠️  Playwright not installed.")
        return [], pending
    print(f"\n🎭 [{time_period}] Phase 3: Playwright scraping {len(pending)} URLs...")
    succeeded, failed = [], []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=HEADERS["User-Agent"],
            extra_http_headers={"Accept-Language": "en-US,en;q=0.5"}
        )
        page = await context.new_page()
        for i, url in enumerate(pending):
            record = await scrape_playwright(url, page, time_period)
            await asyncio.sleep(DELAY_BETWEEN)
            if record.get("error") or not record.get("text") or is_block_page(record):
                failed.append(url)
                reason = "CONSENT WALL" if is_block_page(record) else "FAILED"
                print(f"  ❌ [{i+1}/{len(pending)}] {reason} — {url}")
            else:
                succeeded.append(record)
                done.add(url)
                append_jsonl(record, paths)
                save_checkpoint(done, paths)
                print(f"  ✅ [{i+1}/{len(pending)}] {record['title'] or url[:60]}")
        await browser.close()
    print(f"  Done — {len(succeeded)} succeeded, {len(failed)} failed")
    return succeeded, failed

In [13]:
# ─── Phase 4: Save to Parquet (per sheet) ────────────────────────────────────
 
def build_parquet(paths: dict, time_period: str) -> pd.DataFrame | None:
    print(f"\n💾 [{time_period}] Phase 4: Building Parquet...")
    records = []
    if paths["jsonl"].exists():
        with paths["jsonl"].open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
    if not records:
        print("  ⚠️  No records to save.")
        return None
    df = pd.DataFrame(records)
    df["scraped_at"]  = pd.to_datetime(df["scraped_at"], utc=True, errors="coerce")
    df["text_length"] = pd.to_numeric(df["text_length"], errors="coerce").fillna(0).astype(int)
    df["status"]      = pd.to_numeric(df["status"], errors="coerce")
    if "date_source" not in df.columns:
        df["date_source"] = "missing"
    df.to_parquet(paths["parquet"], index=False, compression="snappy", engine="pyarrow")
    print(f"  ✅ {len(df)} rows → {paths['parquet']}")
    print(f"  Successful: {df['error'].isna().sum()} | Failed: {df['error'].notna().sum()}")
    print(f"  Date source breakdown: {df['date_source'].value_counts().to_dict()}")
    return df

In [14]:
# ─── Main: scrape each sheet independently ────────────────────────────────────
 
all_dfs = {}  # stores resulting DataFrame per sheet
 
for sheet_name, urls in SHEETS.items():
    paths = sheet_paths(sheet_name)
    print(f"\n{'='*60}")
    print(f"  📅 Time period: {sheet_name} ({len(urls)} URLs)")
    print(f"{'='*60}")
 
    done = load_checkpoint(paths)
    if done:
        print(f"  ♻️  Resuming — {len(done)} URLs already done")
 
    # Phase 1 — classify
    classification = await classify_urls(urls, sheet_name)
 
    # Phase 2 — HTTP scrape accessible URLs
    _, http_failed = await run_http_phase(
        classification["ok"], done, paths, sheet_name
    )
 
    # Phase 3 — Playwright for blocked + http-failed
    pw_targets = list(set(classification["blocked"] + classification["error"] + http_failed))
    _, pw_failed = await run_playwright_phase(pw_targets, done, paths, sheet_name)
 
    # Log failed URLs
    with paths["failed"].open("w") as f:
        for u in classification["robots_blocked"]:
            f.write(f"ROBOTS_BLOCKED\t{u}\n")
        for u in pw_failed:
            f.write(f"FAILED\t{u}\n")
 
    # Phase 4 — build Parquet
    df = build_parquet(paths, sheet_name)
    if df is not None:
        all_dfs[sheet_name] = df
 
print(f"\n{'='*60}")
print("  ✅ All time periods done!")
for name, df in all_dfs.items():
    print(f"  {name}: {len(df)} records")
print(f"{'='*60}")


  📅 Time period: 28Mar_10April (64 URLs)

📋 [28Mar_10April] Phase 1: Classifying 64 URLs...
  🚫 429 https://www.yahoo.com/entertainment/music/article/justin-bieber-is-headlining-coachella-this-month-how-the-pop-star-is-forging-a-new-path-163753630.html
  ✅ 200 https://www.theguardian.com/music/2026/apr/10/coachella-2026-justin-bieber-sabrina-carpenter-karol-g
  ✅ 200 https://www.rollingstone.com/music/music-news/justin-bieber-coachella-sneak-preview-1235540010/
  ✅ 200 https://pagesix.com/2026/04/04/celebrity-news/justin-biebers-got-a-lot-riding-on-coachella-comeback/
  ✅ 200 https://eu.desertsun.com/story/entertainment/coachella/2026/04/03/justin-bieber-loves-palm-springs-but-it-doesnt-always-love-him-back-recap-of-infamous-moments/89319620007/
  ✅ 200 https://consequence.net/2026/03/justin-bieber-private-coachella-warmup-show-no-old-songs/
  ✅ 200 https://www.tmz.com/2026/04/08/justin-bieber-coachella-rehearsal-video/
  ✅ 200 https://www.thefader.com/2026/04/08/what-will-justin-bieb